# 10. Securing Our API

##### 10.01.1.0. Skimming: What did you notice and why? Any Questions

_**What?**_  
Currently the `/newsletters` endpoint accepts a newsletter issue as JSON and then sends emails out to all subscribers.  

_**Why?**_  
Only priviledged uses should be able to publish a newsletter (create a newsletter issue) and send it out to subscribers.  
Right now anyone can hit the endpoint and broadcast whatever they want to our existing mailing list.

_**Questions?**_  
What is the best way to breakdown this chapter because it is quite a bulky one?  

While perusing the bulkiest sections are
- Password-based Authentication
- Login
- Sessions + Seed Users

So a session needs to ensure that we have enough context to clear a section without information overload.

## 10.01. Authentication

##### 10.01.1.0. Skimming: What did you notice and why? Any Questions

**What?**  
Authentication Categories, that applications can use to authenticate their users
- Something they know (passwords)
- Something the have (Smartphone, authenticator apps, U2F Keys [Universal Second Factor-NFC, USB])
- Something they are (fingerprints, Face ID)


**Wny?**  
The breakdown is nice, clear, simplifying the general techniques used for authentication.  
Also a bit clear the distinction between
- Authentication - the who
- Authorization - the what

Need for Multi-factor authenticaion also made straightforward because of the flaws of the different authentication techniques when used  
in isolation.

**Questions?**   
None


### 10.01.0. Overview

##### 10.01.0.0.1. Deep Dive: Summarize, ELI5, Connect

### 10.01.1. Drawbacks

### 10.01.2. Multi-factor Authentication

## 10.02. Password-based Authentication

### 10.02.1. Basic Authentication

##### 10.02.0.0. Skimming: What did you notice and why? Any Questions

_**What?**_  
[RFC2617](https://datatracker.ietf.org/doc/html/rfc2617#section-2) and [RFC7617](https://datatracker.ietf.org/doc/html/rfc7617)

Actix makes use of `HttpRequest`. Axum makes use of `Request` but the both use `HeaderMap`.
- Actix -> `actix_web::http::header::HeaderMap`
- Axum -> `http::HeaderMap`

`String` method `strip_prefix` and `splitn`



_**Why?**_  
- First time looking at an RFC
- Similar naming convention
- Cool string methods

_**Questions?**_  
- In the test `request_missing_authorization_are_rejected()` wondered why we aren't using `app.post_newsletters`. 
> Latter we find out that we update `app.post_newsletters` to pass authentication/authoriazation headers. This test needs to return  
> an error, if the appropriate headers are not included.

##### 10.02.0.0.1. Deep Dive: Summarize, ELI5, Connect

#### 10.02.1.1 Extracting Credentials

##### 10.02.1.1.1. Deep Dive: Summarize, ELI5, Connect

### 10.02.2. Password Verification-Naive Approach

##### 10.02.2.0.0. Skimming: What did you notice and why? Any Questions

_**What?**_
`validate_credentials` we use `user_id` of type `Option<_>` mapping it to `Uuid` at the end.  
How we are wiring up a test user with a password without having an explicit `User` type to work with.    
Instatiating the `test_app` then calling `add_test_user` the finally returning `test_app`.  

_**Wny?**_  
Curious why we use `Option<_>` instead of `Option<Uuid>` directly instead?  
Insight into how best to setup a test user for authentication/authorization testing

_**Questions?**_  
Curious why we use `Option<_>` instead of `Option<Uuid>` directly instead?

##### 10.02.2.0.1. Deep Dive: Summarize, ELI5, Connect

### 10.02.3. Password Storage

#### 10.02.3.1. No Need To Store Raw Passwords

##### 10.02.3.1.0. Skimming: What did you notice and why? Any Questions

_**What?**_    
_Injective_
> Meaning for every unique input a function produces a unique output.  
> IF two outputs are the same, their original inputs must have been identical as well.
>
> **Example**  
> $f(x) = 2x + 1$ is an injective function because we get a unique output for every unique input  
> $f(x) = x^2$ is not an injective function because $x$ could be positive or negative and yeild the same output. e.g. 
> $-2$ or $2$

_**Wny?**_  
A password transformation password has to be injective such that if we store the tranformation on the database we can still verify against the  
actually password. i.e. $x \ne y$ then $f(x) \ne f(y)$ 

_**Questions?**_  


##### 10.02.3.1.1. Deep Dive: Summarize, ELI5, Connect

#### 10.02.3.2. Using A Cryptographic Hash

##### 10.02.3.2.0. Skimming: What did you notice and why? Any Questions

_**What?**_  
`format!("{:x}", password_hash)` to get hex-string.  
We add a `TestUser` to generate, store and access a test user that we use with `TestApp` in `spawn_app`


_**Why?**_  
That means to print something in hex we'd run `println!("{val:x}")`.   
Nice clean pattern to generate a test user with a cryptographically hashed password.

_**Questions?**_  
What about binary?  
As I suspected `println!("{val:b}")`

##### 10.02.3.2.1. Deep Dive: Summarize, ELI5, Connect

#### 10.02.3.3. Preimage Attack

##### 10.02.3.2.0. Skimming: What did you notice and why? Any Questions

_**What?**_  
Exponential time complexity ($2^n$) for a brute force attach to match an input string with SHA 256 hash of the password we are trying to hack.  
Where $n$ is the hash length in bits.  
Where $n > 128$ a preimage attack is unfeasible

_**Why?**_  
The name _Preimage attack_. 

_**Questions?**_  
None


##### 10.02.3.3.1. Deep Dive: Summarize, ELI5, Connect

#### 10.02.3.4. Naive Dictionary Attact.

##### 10.02.3.4.0. Skimming: What did you notice and why? Any Questions

_**What?**_  
The math to estimate how long it would take to brute force an alphanumeric password shorter than 17 characters.  
Introduced to the concept of _rainbow tables_. Cool [video](https://www.youtube.com/watch?v=OzVzo9gtiec) and [wiki](https://en.wikipedia.org/wiki/Rainbow_table)

_**Why?**_  
Heard about rainbow tables alot when it comes to securing passwords

_**Questions?**_  
None


##### 10.02.3.4.1. Deep Dive: Summarize, ELI5, Connect

#### 10.02.3.5. Dictionary Attack

##### 10.02.3.5.0. Skimming: What did you notice and why? Any Questions

_**What?**_  
Cryptographic algorithms we discussed until now are designed to be fast. Meaning it is possible to conpute the inverse transform on consumer hardware

_**Why?**_  
Having a cryptographic algorithm that computes hashes fast/effectively is actually a flaw. We need to make the hashes hard/slow to compute inorder to make it
even more difficult for attackers who may have access to dictionaries of hashes who may use rainbow tables.

_**Questions?**_  



##### 10.02.3.5.1. Deep Dive: Summarize, ELI5, Connect

#### 10.02.3.6. Argon2

##### 10.02.3.6.0. Skimming: What did you notice and why? Any Questions

_**What?**_  
- [OWASP](https://cheatsheetseries.owasp.org/index.html)  
- `password_hash` crate

_**Why?**_  
Should be good to revisit and explore the material. We start with the [Password Storage Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Password_Storage_Cheat_Sheet.html)  when looking at  
Argon2. Other cheatsheets also looking quite information rich.

Seems  like we replace what we were doing before with `sha3::SHA_256::digest` with 2 crates, `argon2` and `password_hash`, and we need to include a salt inorder to  
appropriately hash out password.

_**Questions?**_  



##### 10.02.3.6.1. Deep Dive: Summarize, ELI5, Connect

#### 10.02.3.7. Salting

##### 10.02.3.6.0. Skimming: What did you notice and why? Any Questions

_**What?**_
We re-order `validate_credentials`, querying the `user_id`, `password_hash` (_expected_password_) and `salt` first before hashing the password  
got from the request to do a comparison against.

_**Why?**_
Was wondering where we insert a hashed & salted password, and realized that in the handler we are only evaluating the credentials that come with the  
`publish_newsletter` requests are valid.

_**Question?**_  
None


#### 10.02.3.8. PHC String Format

##### 10.02.3.8.0. Skimming: What did you notice and why? Any Questions

_**What?**_
- **PHC** _string_ - Password Hashing Competition String.
- We remove the _salt_ column that we added in the previous section.

_**Why?**_
- Interesting to know the PHC string format is the result of winning a competition on secure password hashing and storage.
- Argon2 takes care of adding a random salt, via the `PasswordHash::new` that returns to us a valid PHC string format password hash.

_**Question?**_  
None


### 10.02.4. Do Not Block The Async Executor

#### 10.02.4.1. Overview

##### 10.02.4.0.0 Skimming: What did you notice and why? Any Questions

_**What?**_  
- Blocking tasks, cooperative scheduling and async tasks.
- Lifetime error we get because of `Password::new(&expected_password_hash)`

_**Wny?**_  
- Seems like blocking tasks are asynchronous tasks that take too long to return a result and therefore are meant to yeild back control to the
  scheduler for other async tasks to make progress without being blocked. Tasks that could block are meant to be run in a separate threadpool
  `tokio::task::spawn_blocking` so that they don't interfere with other async tasks.

- The lifetime issue is somewhat unclear. Would be good to revisit.

_**Questions?**_  
None


##### 10.02.4.0.1. Deep Dive: Summarize, ELI5, Connect

#### 10.02.4.1. Tracing Context Is Thread-Local

##### 10.02.4.1.0 Skimming: What did you notice and why? Any Questions

_**What?**_
- Attaching current span to newly spanwed thread.

_**Why?**_
- We write a custom function in order to do this utilizing `JoinHandle`, `FnOnce`, and trait bound definitions. A nice pattern to learn from.

_**Questions?**_  
None



##### 10.02.4.1.1. Deep Dive: Summarize, ELI5, Connect

### 10.02.5. User Enumeration

##### 10.02.5.0. Skimming: What did you notice and why? Any Questions

_**What?**_
- Timing attacks.
- Realizing that Invalid username and/or password is a good generic way of rejecting both non-existent accounts or password-username mismatch.
  Addressing timing attacks makes it difficult to differentiate between invalid credentials and non-existent credentials.

_**Why?**_
- A clever tactic to identify valid users of an application vs non-existent in Saas applications with specific registered domains as login credentials ( e.g. user@somesaas.com)
-  Imagining a scenario where we know a valid username/email because of timing attacks and initiate a password reset to gain control of the account.

_**Questions?**_  
None



##### 10.02.4.1.1. Deep Dive: Summarize, ELI5, Connect

## 10.03. Is It Safe?

##### 10.03.0.0. Skimming: What did you notice and why? Any Questions

_**What?**_  
- Client Credentials via OAuth2
- Session based authenticaiton
- Identity federation that relies on OpenID connect, an identity layer on top of OAuth2 standard.


_**Why?**_  
- Core concepts in Authentication/Authorization and security.
- 

_**Questions?**_  
- Would a detour to understanding how the OAuth2 standard works be worth it?

##### 10.03.0.0.1. Deep Dive: Summarize, ELI5, Connect

### 10.03.1. Transport Layer Security (TLS)

### 10.03.2. Password Reset.

### 10.03.3. Interaction Types.

### 10.03.4. Machine To Machine.

#### 10.03.4.0. Overview

#### 10.03.4.1. Client Credentials via OAuth2

### 10.03.5. Person Via Browser.

#### 10.03.5.0. Overview

#### 10.03.5.1. Federated Identity

### 10.03.6. Machine to machine, on behalf of a person.

## 10.04. Interlude: Next Steps.

##### 10.01.0.0. Skimming: What did you notice and why? Any Questions

_**What?**_  
- Converting our Basic Authentication to using a Login form with session based authentication

_**Why?**_  
- Looks fun

_**Questions?**_  
- Should we commit to askama now or at the end?
  Tempted to do askama now. But it makes sense to explore it later. Or maybe htmx might be a better exploration here.

##### 10.01.0.0.1. Deep Dive: Summarize, ELI5, Connect

## 10.05. Login Form.

### 10.05.1. Serving HTML Pages

##### 10.05.0.0. Skimming: What did you notice and why? Any Questions

_**What?**_  
- [Internet Is Hard](https://internetingishard.netlify.app/) - For indepth introduction to HTML and CSS
- [Common Rust Lifetimes Misconceptions](https://github.com/pretzelhammer/rust-blog/blob/master/posts/common-rust-lifetime-misconceptions.md#common-rust-lifetime-misconceptions) -
  For a good deep dive into Rust Lifetimes
- [Chrome Dev Tools](https://developer.chrome.com/docs/devtools/open/) & [Firefox Dev Tools](https://firefox-source-docs.mozilla.org/devtools-user/index.html) Documentation

_**Why?**_  
- This book has a treasure trove of cool additional material around Rust, web and software engineering in general.
- All the resources are from the footnotes.

_**Questions?**_  
None


##### 10.05.0.0.1. Deep Dive: Summarize, ELI5, Connect

## 10.06. Login.

### 10.06.0. Overview

##### 10.06.0.0. Skimming: What did you notice and why? Any Questions

##### 10.06.0.0.1. Deep Dive: Summarize, ELI5, Connect

### 10.06.1. HTML Forms

### 10.06.2. Redirect On Success

### 10.06.3.Processing Form Data

### 10.06.4. Contextual Errors

## 10.07. Sessions.

### 10.07.0. Overview

##### 10.07.0.0. Skimming: What did you notice and why? Any Questions

##### 10.07.0.0.1. Deep Dive: Summarize, ELI5, Connect

### 10.07.1. Session-based Authentication

### 10.07.2. Session Store

### 10.07.3. Choosing A Session Store

### 10.07.4. `actix-session`

### 10.07.5. Admin Dashboard

## 10.08. Seed Users.

### 10.08.0. Overview

##### 10.08.0.0. Skimming: What did you notice and why? Any Questions

##### 10.08.0.0.1. Deep Dive: Summarize, ELI5, Connect

### 10.08.1. Database Migration

### 10.08.2. Password Reset.

## 10.09. Refactoring.

### 10.09.0 Overview.

##### 10.09.0.0. Skimming: What did you notice and why? Any Questions

##### 10.09.0.0.1. Deep Dive: Summarize, ELI5, Connect

### 10.09.1. How To Write An `actix-web` middleware

## 10.10. Summary.

##### 10.10.0.0. Skimming: What did you notice and why? Any Questions

##### 10.10.0.0.1. Deep Dive: Summarize, ELI5, Connect